# Compare results: frozen `main` (baseline) vs current `dev`

Loads the distribution pickles produced by **both** codebases and compares them:
a byte/hash summary table, exact numeric diffs, and overlay plots.

**Where the data comes from** — `tests/verify_against_baseline.py` (run it yourself,
~45–60 min) leaves both sets of outputs on disk:

| directory | produced by |
|---|---|
| `outputs/baseline_verify/baseline/` | the **untouched** `main` code (checked out from the locked `baseline` tag) |
| `outputs/baseline_verify/new/` | the **dev** pipeline, same scope |

Re-run the script any time dev changes and you want fresh numbers; this notebook
only ever *reads* those two directories, so you can re-open and compare whenever
you want without re-running anything.

In [ ]:
# OPTIONAL: regenerate both sides now (takes ~45-60 min, streams its progress).
# Uncomment to run from here instead of a terminal:

# import sys
# !{sys.executable} tests/verify_against_baseline.py

In [ ]:
%matplotlib inline
import hashlib, pickle
from pathlib import Path
import pandas as pd

BASELINE_DIR = Path("outputs/baseline_verify/baseline")   # main's outputs
NEW_DIR      = Path("outputs/baseline_verify/new")        # dev's outputs

names = sorted({p.name for p in BASELINE_DIR.glob("*_DISTR_*")}
               | {p.name for p in NEW_DIR.glob("*_DISTR_*")})
if not names:
    print("No outputs found — run  .env/bin/python tests/verify_against_baseline.py  first.")

def _sha(p: Path):
    return hashlib.sha256(p.read_bytes()).hexdigest()[:16] if p.exists() else "MISSING"

rows = []
for n in names:
    ha, hb = _sha(BASELINE_DIR / n), _sha(NEW_DIR / n)
    rows.append({"file": n, "main (baseline) sha256": ha, "dev sha256": hb,
                 "identical": "YES" if ha == hb != "MISSING" else "NO"})
summary = pd.DataFrame(rows)
if len(summary):
    n_ok = (summary["identical"] == "YES").sum()
    print(f"{n_ok}/{len(summary)} files byte-identical"
          + ("  — main and dev agree exactly" if n_ok == len(summary) else "  — INSPECT THE 'NO' ROWS BELOW"))
summary

In [ ]:
import matplotlib.pyplot as plt

def _load_pair(name):
    with open(BASELINE_DIR / name, "rb") as f: a = pickle.load(f)
    with open(NEW_DIR / name, "rb") as f: b = pickle.load(f)
    return a, b

def compare_file(name, lengths=(1, 2), top_n=40):
    """Numeric + visual comparison of one SEQ_DISTR_* or CLS_DISTR_* file.

    Prints support sizes, keys unique to either side, and the maximum
    absolute probability difference over the shared support (0.0 = exact
    agreement), then overlays the top_n sequences (chosen by main's
    probability, filtered to the given sequence lengths).
    """
    a, b = _load_pair(name)
    if name.startswith("SEQ_DISTR_"):
        A = {tuple(s): p for s, p in a[0]}
        B = {tuple(s): p for s, p in b[0]}
        val = lambda d, k: [d[k]]                     # one probability per seq
        ylabel = "P(sequence)"
    else:  # CLS rows: [subseq, counts, probs, total]
        A = {tuple(r[0]): list(r[2]) for r in a}
        B = {tuple(r[0]): list(r[2]) for r in b}
        val = lambda d, k: d[k]                       # one probability per class
        ylabel = "P(class | sequence)"

    common = sorted(set(A) & set(B))
    only_a, only_b = set(A) - set(B), set(B) - set(A)
    max_diff = max((abs(x - y) for k in common
                    for x, y in zip(val(A, k), val(B, k))), default=float("nan"))
    print(f"{name}")
    print(f"  support: main={len(A)}  dev={len(B)}  shared={len(common)}")
    if only_a: print(f"  only in main: {len(only_a)} (e.g. {sorted(only_a)[:3]})")
    if only_b: print(f"  only in dev:  {len(only_b)} (e.g. {sorted(only_b)[:3]})")
    print(f"  max |P_main - P_dev| over shared support = {max_diff}")

    keys = [k for k in common if len(k) in set(lengths)]
    keys.sort(key=lambda k: -val(A, k)[0])
    keys = keys[:top_n]
    if not keys:
        print("  nothing to plot for those lengths")
        return
    n_val = len(val(A, keys[0]))
    fig, ax = plt.subplots(figsize=(14, 5))
    x = range(len(keys))
    w = 0.8 / (2 * n_val)
    cls_lbl = {0: " flat(0)", 1: " up(1)", 2: " down(-1)"} if n_val > 1 else {0: ""}
    for ci in range(n_val):
        ax.bar([xi + (2 * ci) * w for xi in x], [val(A, k)[ci] for k in keys],
               width=w, color=f"C{ci}", label=f"main{cls_lbl.get(ci, ci)}")
        ax.bar([xi + (2 * ci + 1) * w for xi in x], [val(B, k)[ci] for k in keys],
               width=w, color=f"C{ci}", alpha=0.45, hatch="//",
               label=f"dev{cls_lbl.get(ci, ci)}")
    ax.set_xticks(list(x)); ax.set_xticklabels([str(list(k)) for k in keys],
                                               rotation=90, fontsize=7)
    ax.set_ylabel(ylabel); ax.legend(ncol=2)
    ax.set_title(f"{name} — solid = main (frozen), hatched = dev — top {len(keys)} "
                 f"(lengths {sorted(set(lengths))})")
    fig.tight_layout(); plt.show()

def scatter_all(name):
    """Every shared probability from one file as a point: exact agreement = all
    points on the diagonal."""
    a, b = _load_pair(name)
    if name.startswith("SEQ_DISTR_"):
        A = {tuple(s): [p] for s, p in a[0]}; B = {tuple(s): [p] for s, p in b[0]}
    else:
        A = {tuple(r[0]): list(r[2]) for r in a}; B = {tuple(r[0]): list(r[2]) for r in b}
    pts = [(x, y) for k in set(A) & set(B) for x, y in zip(A[k], B[k])]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter([p[0] for p in pts], [p[1] for p in pts], s=4)
    lim = max((max(p) for p in pts), default=1)
    ax.plot([0, lim], [0, lim], "r--", lw=0.8)
    ax.set_xlabel("main (frozen)"); ax.set_ylabel("dev")
    ax.set_title(f"{name}: {len(pts)} probabilities"); plt.show()

In [ ]:
# Detailed look at one sequence distribution and one class distribution.
# Change the file names / lengths / top_n freely; every file in the summary
# table above works.
seq_files = [n for n in names if n.startswith("SEQ_DISTR_")]
cls_files = [n for n in names if n.startswith("CLS_DISTR_")]

if seq_files:
    compare_file(seq_files[0], lengths=(1, 2), top_n=40)
    scatter_all(seq_files[0])
if cls_files:
    compare_file(cls_files[0], lengths=(1,), top_n=16)